In [ ]:
import openai
from openai import OpenAI
from openai.types.beta.threads.message_create_params import Attachment, AttachmentToolFileSearch
import json
import os
import time
from datetime import datetime
import signal
import sys

# Initialize the client based on the configuration
def create_client():
    # These variables are defined at the bottom of the file
    return OpenAI(api_key=OPENAI_API_KEY)

# The client will be initialized after the configuration is loaded
client = None

# Set the default model
default_model = 'gpt-4o-2024-11-20'

# Configuration file paths
CONFIG_FILE = "assistant_config.json"
PROGRESS_FILE = "batch_progress.json"

class GracefulInterruptHandler:
    """Graceful interrupt handler"""
    def __init__(self):
        self.interrupted = False
        signal.signal(signal.SIGINT, self._signal_handler)
        signal.signal(signal.SIGTERM, self._signal_handler)
    
    def _signal_handler(self, signum, frame):
        print("\n⚠️ Interrupt signal received, saving progress...")
        self.interrupted = True

def save_progress(progress_data):
    """Save progress to a file"""
    try:
        with open(PROGRESS_FILE, 'w', encoding='utf-8') as f:
            json.dump(progress_data, f, ensure_ascii=False, indent=2)
        print(f"✅ Progress saved to {PROGRESS_FILE}")
    except Exception as e:
        print(f"❌ Failed to save progress: {e}")

def load_progress():
    """Load progress from a file"""
    if not os.path.exists(PROGRESS_FILE):
        return None
    try:
        with open(PROGRESS_FILE, 'r', encoding='utf-8') as f:
            progress = json.load(f)
        print(f"📂 Found existing progress file, last interrupted at: {progress.get('last_update', 'Unknown time')}")
        return progress
    except Exception as e:
        print(f"⚠️ Failed to read progress file: {e}")
        return None

def get_completed_tasks(output_dir, run_number):
    """Check for completed tasks"""
    completed = set()
    if not os.path.exists(output_dir):
        return completed
    
    for filename in os.listdir(output_dir):
        if filename.endswith(f"_run_{run_number}.txt"):
            # Extract patient ID
            parts = filename.split('_')
            if len(parts) >= 3:
                try:
                    patient_id = int(parts[1])
                    completed.add(patient_id)
                except ValueError:
                    continue
    return completed

def save_config(assistant_id, vector_store_id):
    """Save assistant and vector store configuration"""
    config = {
        "assistant_id": assistant_id,
        "vector_store_id": vector_store_id
    }
    with open(CONFIG_FILE, 'w', encoding='utf-8') as f:
        json.dump(config, f, ensure_ascii=False, indent=2)
    print(f"✅ Configuration saved to {CONFIG_FILE}")

def load_config():
    """Load configuration, return None if it does not exist"""
    if not os.path.exists(CONFIG_FILE):
        return None
    try:
        with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
            config = json.load(f)
        # Verify if the assistant and vector store still exist
        try:
            client.beta.assistants.retrieve(config["assistant_id"])
            client.beta.vector_stores.retrieve(config["vector_store_id"])
            print(f"✅ Found existing configuration: Assistant ID {config['assistant_id']}, Vector Store ID {config['vector_store_id']}")
            return config
        except Exception as e:
            print("Existing configuration is invalid, will recreate: " + str(e))
            print("Error type: " + type(e).__name__)
            return None
    except Exception as e:
        print("Failed to read configuration file: " + str(e))
        return None

def create_assistant():
    """Create a medical assistant"""
    assistant = client.beta.assistants.create(
        model=default_model,
        instructions="""
You are a medical assistant specializing in the diagnosis and treatment of intracranial hemorrhage (ICH). Your task is to provide structured, precise clinical recommendations **based solely on CT imaging data**, which will be provided in a structured JSON format. You do not have access to any additional clinical data.

You are knowledgeable about the following principles for identifying causes:
- Deep intraparenchymal hemorrhage → Spontaneous ICH
- Subdural or epidural hemorrhage → Traumatic brain injury
- Isolated subarachnoid hemorrhage (SAH), without other types of hemorrhage → Aneurysmal hemorrhage

When formulating recommendations, adhere to the following clinical guidelines and trial results:
- **Spontaneous ICH**: 2022 AHA/ASA Guideline for Spontaneous Intracerebral Hemorrhage, and clinical trials including ENRICH, INTERACT3, INTERACT4, SWITCH, ANNEXA-I.
- **Traumatic brain injury**: 2018 Brain Trauma Foundation Guideline – "Management of Severe Traumatic Brain Injury (First 24 Hours)"
- **Aneurysmal SAH**: 2023 AHA/ASA Guideline for Aneurysmal Subarachnoid Hemorrhage

---

### Step 1: Determine Hemorrhage Cause

Based on the CT findings (location and type of hemorrhage), identify the most likely cause from:
- Spontaneous ICH
- Traumatic brain injury
- Aneurysmal hemorrhage

---

### Step 2: Generate Recommendations

Based on the CT findings, hemorrhage cause, and relevant guidelines, provide:

1. **Additional Examinations**
   - Identify further tests that help complete the diagnostic process, especially focusing on aneurysm detection if applicable.
   - Clearly explain the **purpose** of each test.
   - Specify the **guideline or clinical trial** source supporting each test.

2. **Treatments**
   - Provide specific treatment suggestions (e.g., medical therapy, surgical intervention, BP control).
   - Each treatment should include a **detailed description** of what to do and **why** it is appropriate based on CT findings and the hemorrhage type.
   - Specify the **guideline or trial source** for each recommendation.

---

### Step 3: Safety Note for SAH

If SAH volume is very small (>0 mL, ≤1 mL) OR SAH confidence is low (<0.5), include the following warning in the output:

> "The segmented SAH finding is of very small volume or low confidence. Clinical correlation and, if necessary, further evaluation are recommended."

---

### Output Format

Respond with both:

#### 1. A JSON object:

```json
{
  "Cause": ,
  "Examinations": [
    {
      "Name": "Test A",
      "Purpose": "Why test A",
      "Source": 
    }
  ],
  "Treatments": [
    {
      "Name": "Treatment A",
      "Purpose": "Detailed treatment plan: what to do and why, based on CT findings",
      "Source": 
    }
  ]
}

2. A **human-readable markdown table** including:

### Examination Recommendations

| Name | Purpose | Source |

### Treatment Recommendations

| Name | Purpose (Detailed Treatment Plan) | Source |
""",
        tools=[{"type": "file_search"}],
        name='ICH Assistant',
    )
    return assistant

def upload_guidelines():
    """Upload PDF guideline files and bind to vector search"""
    try:
        vector_store = client.beta.vector_stores.create(name="guidelines")
        print(f"✅ Vector store created successfully, ID: {vector_store.id}")
        
        # Use the current working directory
        pdf_dir = "pdf"
        file_paths = [
            os.path.join(pdf_dir, "Clinical_Trials.pdf"),
            os.path.join(pdf_dir, "2018_TBI_Guideline.pdf"),
            os.path.join(pdf_dir, "2022_ICH_Guideline.pdf"),
            os.path.join(pdf_dir, "2023_aSAH_Guideline.pdf")
        ]
        
        pdf_files = []
        for path in file_paths:
            try:
                with open(path, "rb") as f:
                    pdf_files.append((os.path.basename(path), f.read()))
                    print(f"✅ Successfully read file: {path}")
            except FileNotFoundError:
                print(f"⚠️ Warning: PDF file not found: {path}")
                raise Exception(f"Necessary PDF file not found: {path}")
        
        if not pdf_files:
            raise Exception("No PDF files available for upload")
        
        file_batch = client.beta.vector_stores.file_batches.upload_and_poll(
            vector_store_id=vector_store.id,
            files=[(name, content) for name, content in pdf_files]
        )
        
        if file_batch.status != "completed":
            raise Exception(f"File upload did not complete, status: {file_batch.status}")
        
        print(f"✅ File upload status: {file_batch.status}")
        print(f"✅ File count: {file_batch.file_counts}")
        
        print("Waiting for vector store to initialize...")
        time.sleep(5)
        
        vs_check = client.beta.vector_stores.retrieve(vector_store_id=vector_store.id)
        print(f"✅ Vector store validation successful: {vs_check.id}")
        return vector_store
            
    except Exception as e:
        print(f"❌ Vector store creation or file upload failed: {e}")
        raise

def setup_assistant():
    """Set up the assistant (create or load existing)"""
    config = load_config()
    
    if config:
        print("🔄 Using existing assistant configuration")
        try:
            # Validate if the configuration is still valid
            print("Validating assistant configuration...")
            assistant = client.beta.assistants.retrieve(config["assistant_id"])
            print(f"✅ Assistant validation successful: {assistant.name}")
            
            print("Validating vector store...")
            vector_store = client.beta.vector_stores.retrieve(config["vector_store_id"])
            print(f"✅ Vector store validation successful: {vector_store.name}")
            
            return config["assistant_id"], config["vector_store_id"]
        except Exception as e:
            print(f"❌ Configuration validation failed: {e}")
            print(f"Error type: {type(e).__name__}")
            if hasattr(e, 'response'):
                print(f"HTTP status code: {e.response.status_code if hasattr(e.response, 'status_code') else 'Unknown'}")
            print("Recreating assistant...")
    
    print("🆕 Creating new assistant and vector store")
    try:
        print("Step 1: Creating assistant...")
        assistant = create_assistant()
        print("✅ Assistant creation complete")
        
        print("Step 2: Uploading guideline files...")
        vector_store = upload_guidelines()
        print("✅ Guideline upload complete")
        
        print("Step 3: Binding vector search...")
        assistant = client.beta.assistants.update(
            assistant_id=assistant.id,
            tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}}
        )
        print(f"✅ Assistant successfully bound to vector search ID: {vector_store.id}")
        
        save_config(assistant.id, vector_store.id)
        return assistant.id, vector_store.id
        
    except Exception as e:
        print(f"❌ Assistant setup failed: {e}")
        print(f"Error type: {type(e).__name__}")
        print(f"Error details: {str(e)}")
        if hasattr(e, 'response'):
            print(f"HTTP status code: {e.response.status_code if hasattr(e.response, 'status_code') else 'Unknown'}")
            if hasattr(e.response, 'text'):
                print(f"Response content: {e.response.text}")
        raise

def check_rate_limits():
    """Check OpenAI API usage limits"""
    try:
        # Send a simple request to get limit information from the response headers
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": "test"}],
            max_tokens=1
        )
        
        # OpenAI may return limit information in headers, but it's not directly accessible via the SDK
        print("✅ API connection normal")
        print("📊 Rate Limits Info:")
        print("  - Please check the OpenAI Dashboard for detailed limit information")
        print("  - Link: https://platform.openai.com/usage")
        return True
        
    except Exception as e:
        print(f"❌ Failed to check rate limits: {e}")
        return False

def estimate_processing_time(num_patients, num_runs):
    """Estimate processing time"""
    # Estimate processing time per patient (including API calls and delays)
    avg_processing_time = 30  # Assuming an average of 30 seconds per patient
    delay_per_patient = DELAY_BETWEEN_PATIENTS
    delay_between_runs = DELAY_BETWEEN_RUNS
    
    time_per_run = num_patients * (avg_processing_time + delay_per_patient)
    total_delay_between_runs = (num_runs - 1) * delay_between_runs
    total_time = (time_per_run * num_runs) + total_delay_between_runs
    
    hours = total_time // 3600
    minutes = (total_time % 3600) // 60
    
    print(f"⏱️ Estimated processing time:")
    print(f"  - Per run: ~{time_per_run//60}min{time_per_run%60}s")
    print(f"  - Total: ~{hours}h{minutes}min")
    print(f"  - Patients: {num_patients}, Runs: {num_runs}")

def write_error_log(error_log_file, context, error_msg):
    """Write to error log"""
    with open(error_log_file, "a", encoding="utf-8") as log_file:
        log_file.write(f"Context: {context}, Error message: {error_msg}\n")

def process_single_patient(assistant_id, patient, run_number, output_dir, error_log_file, interrupt_handler, max_retries=3):
    """Process a single patient (with retry support)"""
    patient_id = patient["ID"]
    
    for attempt in range(max_retries):
        # Check if interrupted
        if interrupt_handler.interrupted:
            print(f"⚠️ Interrupt signal detected, stopping processing for patient {patient_id}")
            return False
        
        if attempt > 0:
            print(f"🔄 [Run {run_number}] Retrying patient {patient_id} (attempt {attempt+1}/{max_retries})")
        else:
            print(f"🧠 [Run {run_number}] Processing patient ID: {patient_id}")
        
        prompt = f"""Below is the CT image analysis for a patient:
1. Intraparenchymal Hemorrhage:
   - Volume: {patient['Intraparenchymal_Hemorrhage']['Volume']} ml
   - Location: {patient['Intraparenchymal_Hemorrhage']['Location']}
2. Intraventricular Hemorrhage:
   - Volume: {patient['Intraventricular_Hemorrhage']['Volume']} ml
   - Location: {patient['Intraventricular_Hemorrhage']['Location']}
3. Perihematomal Edema:
   - Volume: {patient['Perihematomal_Edema']['Volume']} ml
   - Location: {patient['Perihematomal_Edema']['Location']}
4. Subarachnoid Hemorrhage:
   - Volume: {patient['Subarachnoid_Hemorrhage']['Volume']} ml
   - Location: {patient['Subarachnoid_Hemorrhage']['Location']}
5. Subdural Hemorrhage:
   - Volume: {patient['Subdural_Hemorrhage']['Volume']} ml
   - Location: {patient['Subdural_Hemorrhage']['Location']}
   - Confidence: {patient['Subarachnoid_Hemorrhage']['Confidence']}
6. Epidural Hemorrhage:
   - Volume: {patient['Epidural_Hemorrhage']['Volume']} ml
   - Location: {patient['Epidural_Hemorrhage']['Location']}
7. Hydrocephalus: {patient['Hydrocephalus']}
8. Midline Shift: {patient['Midline_Shift']}
"""
        try:
            thread = client.beta.threads.create()
            client.beta.threads.messages.create(
                thread_id=thread.id,
                role='user',
                content=prompt
            )

            run = client.beta.threads.runs.create_and_poll(
                thread_id=thread.id,
                assistant_id=assistant_id,
                timeout=600,
                temperature=0
            )

            if run.status != "completed":
                error_info = f'Run failed with status: {run.status}'
                
                # Add more run status information
                if hasattr(run, 'last_error') and run.last_error:
                    error_info += f' | Last error: {run.last_error}'
                
                if hasattr(run, 'failed_at') and run.failed_at:
                    error_info += f' | Failed at: {run.failed_at}'
                
                if hasattr(run, 'required_action') and run.required_action:
                    error_info += f' | Required action: {run.required_action}'
                
                # Try to get detailed run information
                try:
                    run_details = client.beta.threads.runs.retrieve(
                        thread_id=thread.id,
                        run_id=run.id
                    )
                    if hasattr(run_details, 'last_error') and run_details.last_error:
                        error_info += f' | Detailed error: {run_details.last_error}'
                except:
                    pass
                
                raise Exception(error_info)

            messages_cursor = client.beta.threads.messages.list(thread_id=thread.id, run_id=run.id)
            messages = [message for message in messages_cursor]
            treatment_plan = messages[0].content[0].text.value

            output_file = os.path.join(output_dir, f"patient_{patient_id}_report_run_{run_number}.txt")
            with open(output_file, "w", encoding="utf-8") as file:
                file.write(treatment_plan)
            print(f"✅ Saved to {output_file}")
            return True

        except Exception as ex:
            error_details = []
            error_details.append(f"Error type: {type(ex).__name__}")
            error_details.append(f"Error message: {str(ex)}")
            
            # If it is an OpenAI API error, provide more details
            if hasattr(ex, 'response'):
                if hasattr(ex.response, 'status_code'):
                    error_details.append(f"HTTP status code: {ex.response.status_code}")
                if hasattr(ex.response, 'text'):
                    error_details.append(f"Response content: {ex.response.text[:200]}...")
            
            # If there is an error code
            if hasattr(ex, 'code'):
                error_details.append(f"Error code: {ex.code}")
            
            # If there is a message body
            if hasattr(ex, 'body'):
                error_details.append(f"Error body: {ex.body}")
            
            error_msg = f"Attempt {attempt+1}/{max_retries} - " + " | ".join(error_details)
            print(f"❌ Error for {patient_id} (attempt {attempt+1}):")
            for detail in error_details:
                print(f"   {detail}")
            
            # If not the last attempt, wait and retry
            if attempt < max_retries - 1:
                wait_time = RETRY_DELAY_BASE * (attempt + 1)  # Incremental wait time
                print(f"⏳ Waiting for {wait_time} seconds before retrying...")
                time.sleep(wait_time)
            else:
                # Last attempt failed, writing to error log
                write_error_log(error_log_file, f"{patient_id}_run_{run_number}", error_msg)
                print(f"❌ Failed to process patient {patient_id}, max retries reached")
    
    return False

def process_patients(assistant_id, input_file, output_dir, run_number=1, interrupt_handler=None, resume_data=None):
    """Process patient data (with resume support)"""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    error_log_file = os.path.join(output_dir, f"error_log_run_{run_number}.txt")
    
    try:
        with open(input_file, 'r', encoding='utf-8') as file:
            data = json.load(file)
    except Exception as e:
        print(f"Could not read patient data file: {e}")
        return False
    
    # If in resume mode, check for completed tasks
    completed_patients = set()
    if resume_data:
        completed_patients = set(resume_data.get('completed_patients', []))
        print(f"📋 Resume mode: Skipping {len(completed_patients)} completed patients")
    
    total_patients = len(data)
    processed = 0
    
    for i, patient in enumerate(data):
        patient_id = patient["ID"]
        
        # Check if output file already exists (skip if it does)
        output_file = os.path.join(output_dir, f"patient_{patient_id}_report_run_{run_number}.txt")
        if os.path.exists(output_file):
            print(f"⏭️  Skipping completed patient {patient_id} (file exists)")
            completed_patients.add(patient_id)
            processed += 1
            continue
        
        # Process patient
        success = process_single_patient(assistant_id, patient, run_number, output_dir, error_log_file, interrupt_handler, MAX_RETRIES)
        
        if success:
            completed_patients.add(patient_id)
            processed += 1
            
            # Save progress every 5 patients
            if processed % 5 == 0 or interrupt_handler.interrupted:
                progress_data = {
                    'run_number': run_number,
                    'total_patients': total_patients,
                    'processed_patients': processed,
                    'completed_patients': list(completed_patients),
                    'last_update': datetime.now().isoformat(),
                    'status': 'interrupted' if interrupt_handler.interrupted else 'running'
                }
                save_progress(progress_data)
        
        # Check for interrupt signal
        if interrupt_handler.interrupted:
            print(f"\n⚠️ Task interrupted, {processed}/{total_patients} patients processed")
            progress_data = {
                'run_number': run_number,
                'total_patients': total_patients,
                'processed_patients': processed,
                'completed_patients': list(completed_patients),
                'last_update': datetime.now().isoformat(),
                'status': 'interrupted'
            }
            save_progress(progress_data)
            return False
        
        # Brief pause to avoid API rate limits
        if i < len(data) - 1:
            time.sleep(DELAY_BETWEEN_PATIENTS)
    
    # Clean up progress file upon completion
    if os.path.exists(PROGRESS_FILE):
        os.remove(PROGRESS_FILE)
        print("✅ Task complete, progress file cleaned up")
    
    return True

def batch_test_with_resume(num_runs=10, resume=False):
    """Batch testing function (with resume support)"""
    interrupt_handler = GracefulInterruptHandler()
    
    try:
        assistant_id, vector_store_id = setup_assistant()
        
        # Use the current working directory
        input_file = os.path.join("json", JSON_FILENAME)
        output_dir = "batch_results"
        
        start_run = 1
        resume_data = None
        
        # Check if resume is needed
        if resume:
            progress = load_progress()
            if progress:
                start_run = progress['run_number']
                resume_data = progress
                print(f"🔄 Resuming from run {start_run}, {progress['processed_patients']}/{progress['total_patients']} patients processed")
            else:
                print("⚠️ No resumable progress found, starting from the beginning")
        
        print(f"\n🚀 Starting batch test - from run {start_run}, {num_runs} runs in total")
        
        for run in range(start_run, num_runs + 1):
            print(f"\n{'='*50}")
            print(f"🔄 Starting run {run}/{num_runs}")
            print(f"{'='*50}")
            
            # If this is the first run in resume mode, use resume data
            current_resume_data = resume_data if (run == start_run and resume_data) else None
            
            success = process_patients(assistant_id, input_file, output_dir, run, interrupt_handler, current_resume_data)
            
            if not success:
                if interrupt_handler.interrupted:
                    print(f"\n⚠️ Task was interrupted during run {run}")
                    print(f"💡 Use the --resume parameter to resume execution")
                else:
                    print(f"❌ Run {run} failed")
                return
            
            print(f"✅ Run {run} complete")
            
            # Brief pause to avoid API rate limits
            if run < num_runs:
                print(f"⏱️ Waiting {DELAY_BETWEEN_RUNS} seconds before starting the next run...")
                time.sleep(DELAY_BETWEEN_RUNS)
        
        print(f"\n🎉 All {num_runs} test runs complete!")
        print(f"📁 Results saved in: {output_dir}")
        
    except Exception as e:
        print(f"❌ Batch test failed: {e}")
        # Save error state
        if interrupt_handler.interrupted:
            print("💡 Use the --resume parameter to attempt to resume execution")

def main():
    """Main function"""
    global client
    
    # Initialize the client
    client = create_client()
    
    print("Batch Test Configuration:")
    print(f"  - JSON file: {JSON_FILENAME}")
    print(f"  - Number of runs: {NUM_RUNS}")
    print(f"  - Max retries: {MAX_RETRIES}")
    print(f"  - Auto-resume: {'Enabled' if AUTO_RESUME else 'Disabled'}")
    print(f"  - Delay between patients: {DELAY_BETWEEN_PATIENTS}s")
    print(f"  - Delay between runs: {DELAY_BETWEEN_RUNS}s")
    print(f"  - Retry delay: {RETRY_DELAY_BASE}s incremental")
    print()
    
    # Check API connection and limits
    print("🔍 Checking API connection...")
    if check_rate_limits():
        print()
    
    # Estimate processing time
    try:
        input_file = os.path.join("json", JSON_FILENAME)
        with open(input_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        estimate_processing_time(len(data), NUM_RUNS)
        print()
    except Exception as e:
        print(f"⚠️ Could not estimate processing time: {e}")
        print()
    
    # Check if resume is needed
    resume = False
    if AUTO_RESUME:
        progress = load_progress()
        if progress and progress.get('status') == 'interrupted':
            resume = True
            print("Detected an incomplete task, will auto-resume")
    
    batch_test_with_resume(NUM_RUNS, resume)

# === Configuration Area - Modify parameters here ===
OPENAI_API_KEY = "your key"
JSON_FILENAME = "patients_data_other.json"  # JSON filename (place in the json folder)
NUM_RUNS = 10          # Number of runs
MAX_RETRIES = 3        # Max retries per patient
AUTO_RESUME = True    # Automatically detect and resume interrupted tasks

# === Delay Settings ===
DELAY_BETWEEN_PATIENTS = 5  # Delay between patients (seconds) - to adapt to RPM limits
DELAY_BETWEEN_RUNS = 5      # Delay between runs (seconds)
RETRY_DELAY_BASE = 3        # Base retry delay (seconds), incremental: 3s, 6s, 9s
# ==============================

if __name__ == "__main__":
    main()


批量测试配置:
  - 使用API: 原生OpenAI API
  - JSON文件: patients_data_other.json
  - 运行次数: 10
  - 重试次数: 3
  - 自动恢复: 开启
  - 患者间延迟: 5秒
  - 运行间延迟: 5秒
  - 重试延迟: 3秒递增

🔍 检查API连接...
✅ API连接正常
📊 Rate Limits信息:
  - 请查看OpenAI Dashboard获取详细限制信息
  - 链接: https://platform.openai.com/usage

⏱️ 预估处理时间:
  - 每轮: ~120分45秒
  - 总计: ~20小时8分钟
  - 患者数: 207, 运行轮数: 10

📂 找到现有进度文件，上次中断于: 2025-08-25T19:25:10.751770
检测到未完成的任务，将自动恢复
✅ 找到现有配置: 助手ID asst_h2SR2nQrrjExxTbCGXmg3Lmd, 向量存储ID vs_68a9cb65af248191a07664cfed890593
🔄 使用现有助手配置
验证助手配置...
✅ 助手验证成功: ICH Assistant
验证向量存储...
✅ 向量存储验证成功: guidelines
📂 找到现有进度文件，上次中断于: 2025-08-25T19:25:10.751770
🔄 从第 3 次运行恢复，已处理 134/207 个患者

🚀 开始批量测试 - 从第 3 次运行开始，共 10 次

🔄 开始第 3/10 次运行
📋 恢复模式：跳过已完成的 134 个患者
⏭️  跳过已完成的患者 17 (文件已存在)
⏭️  跳过已完成的患者 19 (文件已存在)
⏭️  跳过已完成的患者 22 (文件已存在)
⏭️  跳过已完成的患者 27 (文件已存在)
⏭️  跳过已完成的患者 28 (文件已存在)
⏭️  跳过已完成的患者 33 (文件已存在)
⏭️  跳过已完成的患者 35 (文件已存在)
⏭️  跳过已完成的患者 40 (文件已存在)
⏭️  跳过已完成的患者 41 (文件已存在)
⏭️  跳过已完成的患者 43 (文件已存在)
⏭️  跳过已完成的患者 48 (文件已存在)
⏭️  跳过已完成的患者 50 (文件已存在)
⏭️  跳过已

✅ Saved to batch_results\patient_429_report_run_3.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run 3] Processing patient ID: 430
✅ Saved to batch_results\patient_430_report_run_3.txt
🧠 [Run 3] Processing patient ID: 432
✅ Saved to batch_results\patient_432_report_run_3.txt
🧠 [Run 3] Processing patient ID: 434
✅ Saved to batch_results\patient_434_report_run_3.txt
🧠 [Run 3] Processing patient ID: 435
✅ Saved to batch_results\patient_435_report_run_3.txt
🧠 [Run 3] Processing patient ID: 438
✅ Saved to batch_results\patient_438_report_run_3.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run 3] Processing patient ID: 443
✅ Saved to batch_results\patient_443_report_run_3.txt
⏭️  跳过已完成的患者 446 (文件已存在)
🧠 [Run 3] Processing patient ID: 447
✅ Saved to batch_results\patient_447_report_run_3.txt
🧠 [Run 3] Processing patient ID: 449
✅ Saved to batch_results\patient_449_report_run_3.txt
🧠 [Run 3] Processing patient ID: 452
✅ Saved to batch_results\patient_452_report_run_3.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run 3] Proce

✅ Saved to batch_results\patient_155_report_run_4.txt
🧠 [Run 4] Processing patient ID: 157
✅ Saved to batch_results\patient_157_report_run_4.txt
🧠 [Run 4] Processing patient ID: 159
✅ Saved to batch_results\patient_159_report_run_4.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run 4] Processing patient ID: 160
✅ Saved to batch_results\patient_160_report_run_4.txt
🧠 [Run 4] Processing patient ID: 166
✅ Saved to batch_results\patient_166_report_run_4.txt
🧠 [Run 4] Processing patient ID: 169
✅ Saved to batch_results\patient_169_report_run_4.txt
🧠 [Run 4] Processing patient ID: 173
✅ Saved to batch_results\patient_173_report_run_4.txt
🧠 [Run 4] Processing patient ID: 181
✅ Saved to batch_results\patient_181_report_run_4.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run 4] Processing patient ID: 182
✅ Saved to batch_results\patient_182_report_run_4.txt
🧠 [Run 4] Processing patient ID: 183
✅ Saved to batch_results\patient_183_report_run_4.txt
🧠 [Run 4] Processing patient ID: 186
✅ Saved to batch_results\patient

🧠 [Run 4] Processing patient ID: 364
✅ Saved to batch_results\patient_364_report_run_4.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run 4] Processing patient ID: 369
✅ Saved to batch_results\patient_369_report_run_4.txt
🧠 [Run 4] Processing patient ID: 371
✅ Saved to batch_results\patient_371_report_run_4.txt
🧠 [Run 4] Processing patient ID: 376
✅ Saved to batch_results\patient_376_report_run_4.txt
🧠 [Run 4] Processing patient ID: 377
✅ Saved to batch_results\patient_377_report_run_4.txt
🧠 [Run 4] Processing patient ID: 378
✅ Saved to batch_results\patient_378_report_run_4.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run 4] Processing patient ID: 379
✅ Saved to batch_results\patient_379_report_run_4.txt
🧠 [Run 4] Processing patient ID: 380
✅ Saved to batch_results\patient_380_report_run_4.txt
🧠 [Run 4] Processing patient ID: 381
✅ Saved to batch_results\patient_381_report_run_4.txt
🧠 [Run 4] Processing patient ID: 382
✅ Saved to batch_results\patient_382_report_run_4.txt
⏭️  跳过已完成的患者 383 (文件已存在)
⏭️  跳过已

🧠 [Run 5] Processing patient ID: 73
✅ Saved to batch_results\patient_73_report_run_5.txt
⏭️  跳过已完成的患者 75 (文件已存在)
🧠 [Run 5] Processing patient ID: 80
✅ Saved to batch_results\patient_80_report_run_5.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run 5] Processing patient ID: 84
✅ Saved to batch_results\patient_84_report_run_5.txt
🧠 [Run 5] Processing patient ID: 86
✅ Saved to batch_results\patient_86_report_run_5.txt
🧠 [Run 5] Processing patient ID: 87
✅ Saved to batch_results\patient_87_report_run_5.txt
🧠 [Run 5] Processing patient ID: 90
✅ Saved to batch_results\patient_90_report_run_5.txt
🧠 [Run 5] Processing patient ID: 92
✅ Saved to batch_results\patient_92_report_run_5.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run 5] Processing patient ID: 95
✅ Saved to batch_results\patient_95_report_run_5.txt
🧠 [Run 5] Processing patient ID: 106
✅ Saved to batch_results\patient_106_report_run_5.txt
🧠 [Run 5] Processing patient ID: 107
✅ Saved to batch_results\patient_107_report_run_5.txt
🧠 [Run 5] Processing pat

...

✅ Saved to batch_results\patient_111_report_run_9.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run 9] Processing patient ID: 113
✅ Saved to batch_results\patient_113_report_run_9.txt
🧠 [Run 9] Processing patient ID: 114
✅ Saved to batch_results\patient_114_report_run_9.txt
🧠 [Run 9] Processing patient ID: 115
✅ Saved to batch_results\patient_115_report_run_9.txt
🧠 [Run 9] Processing patient ID: 117
✅ Saved to batch_results\patient_117_report_run_9.txt
🧠 [Run 9] Processing patient ID: 118
✅ Saved to batch_results\patient_118_report_run_9.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run 9] Processing patient ID: 121
✅ Saved to batch_results\patient_121_report_run_9.txt
🧠 [Run 9] Processing patient ID: 125
✅ Saved to batch_results\patient_125_report_run_9.txt
🧠 [Run 9] Processing patient ID: 126
✅ Saved to batch_results\patient_126_report_run_9.txt
🧠 [Run 9] Processing patient ID: 128
✅ Saved to batch_results\patient_128_report_run_9.txt
🧠 [Run 9] Processing patient ID: 129
✅ Saved to batch_results\patient

✅ Saved to batch_results\patient_303_report_run_9.txt
🧠 [Run 9] Processing patient ID: 305
✅ Saved to batch_results\patient_305_report_run_9.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run 9] Processing patient ID: 310
✅ Saved to batch_results\patient_310_report_run_9.txt
🧠 [Run 9] Processing patient ID: 312
✅ Saved to batch_results\patient_312_report_run_9.txt
🧠 [Run 9] Processing patient ID: 314
✅ Saved to batch_results\patient_314_report_run_9.txt
🧠 [Run 9] Processing patient ID: 316
✅ Saved to batch_results\patient_316_report_run_9.txt
🧠 [Run 9] Processing patient ID: 318
✅ Saved to batch_results\patient_318_report_run_9.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run 9] Processing patient ID: 325
✅ Saved to batch_results\patient_325_report_run_9.txt
🧠 [Run 9] Processing patient ID: 326
✅ Saved to batch_results\patient_326_report_run_9.txt
🧠 [Run 9] Processing patient ID: 328
✅ Saved to batch_results\patient_328_report_run_9.txt
🧠 [Run 9] Processing patient ID: 334
✅ Saved to batch_results\patient

✅ Saved to batch_results\patient_483_report_run_9.txt
🧠 [Run 9] Processing patient ID: 485
✅ Saved to batch_results\patient_485_report_run_9.txt
🧠 [Run 9] Processing patient ID: 486
✅ Saved to batch_results\patient_486_report_run_9.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run 9] Processing patient ID: 488
✅ Saved to batch_results\patient_488_report_run_9.txt
🧠 [Run 9] Processing patient ID: 489
✅ Saved to batch_results\patient_489_report_run_9.txt
✅ 任务完成，已清理进度文件
✅ 第 9 次运行完成
⏱️ 等待5秒后开始下一次运行...

🔄 开始第 10/10 次运行
🧠 [Run 10] Processing patient ID: 17
✅ Saved to batch_results\patient_17_report_run_10.txt
🧠 [Run 10] Processing patient ID: 19
✅ Saved to batch_results\patient_19_report_run_10.txt
🧠 [Run 10] Processing patient ID: 22
✅ Saved to batch_results\patient_22_report_run_10.txt
🧠 [Run 10] Processing patient ID: 27
✅ Saved to batch_results\patient_27_report_run_10.txt
🧠 [Run 10] Processing patient ID: 28
✅ Saved to batch_results\patient_28_report_run_10.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run

🧠 [Run 10] Processing patient ID: 208
✅ Saved to batch_results\patient_208_report_run_10.txt
🧠 [Run 10] Processing patient ID: 210
✅ Saved to batch_results\patient_210_report_run_10.txt
🧠 [Run 10] Processing patient ID: 211
✅ Saved to batch_results\patient_211_report_run_10.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run 10] Processing patient ID: 213
✅ Saved to batch_results\patient_213_report_run_10.txt
🧠 [Run 10] Processing patient ID: 215
✅ Saved to batch_results\patient_215_report_run_10.txt
🧠 [Run 10] Processing patient ID: 216
✅ Saved to batch_results\patient_216_report_run_10.txt
🧠 [Run 10] Processing patient ID: 217
✅ Saved to batch_results\patient_217_report_run_10.txt
🧠 [Run 10] Processing patient ID: 219
✅ Saved to batch_results\patient_219_report_run_10.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run 10] Processing patient ID: 221
✅ Saved to batch_results\patient_221_report_run_10.txt
🧠 [Run 10] Processing patient ID: 224
✅ Saved to batch_results\patient_224_report_run_10.txt
🧠 [Run 10] P

✅ Saved to batch_results\patient_396_report_run_10.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run 10] Processing patient ID: 397
✅ Saved to batch_results\patient_397_report_run_10.txt
🧠 [Run 10] Processing patient ID: 399
✅ Saved to batch_results\patient_399_report_run_10.txt
🧠 [Run 10] Processing patient ID: 400
✅ Saved to batch_results\patient_400_report_run_10.txt
🧠 [Run 10] Processing patient ID: 402
✅ Saved to batch_results\patient_402_report_run_10.txt
🧠 [Run 10] Processing patient ID: 403
✅ Saved to batch_results\patient_403_report_run_10.txt
✅ 进度已保存到 batch_progress.json
🧠 [Run 10] Processing patient ID: 404
✅ Saved to batch_results\patient_404_report_run_10.txt
🧠 [Run 10] Processing patient ID: 406
✅ Saved to batch_results\patient_406_report_run_10.txt
🧠 [Run 10] Processing patient ID: 408
✅ Saved to batch_results\patient_408_report_run_10.txt
🧠 [Run 10] Processing patient ID: 411
✅ Saved to batch_results\patient_411_report_run_10.txt
🧠 [Run 10] Processing patient ID: 412
✅ Saved to b